# FinMiniLM Eval — Base vs Fine-Tuned Comparison

## Approach: InformationRetrievalEvaluator (Industry Standard)

This notebook evaluates the base `all-MiniLM-L6-v2` against the fine-tuned
`minilm_finetuned` model using `sentence-transformers`'
`InformationRetrievalEvaluator` — the standard evaluation framework for
dense retrieval benchmarks (BEIR, MTEB).

### Metrics reported

| Metric | Meaning |
|---|---|
| **NDCG@10** | Normalised Discounted Cumulative Gain — main ranking quality metric |
| **MRR** | Mean Reciprocal Rank — how high the first correct result appears |
| **Recall@10** | % of true positives found in the top-10 results |

### Eval set construction

We use a **held-out 20% stratified split** of the Groq-generated pairs from
`training_pairs.csv`. The full S4 corpus acts as the retrieval pool, making
this a realistic open-domain retrieval evaluation over all 41 PDFs.

In [ ]:
import sys, os
from pathlib import Path

# ── Find repo root by walking up until config.py is found ─────────────────────
def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for parent in [p] + list(p.parents):
        if (parent / 'config.py').exists():
            return parent
    raise FileNotFoundError('config.py not found — cannot locate repo root')

REPO_ROOT = find_repo_root()
print(f'Repo root: {REPO_ROOT}')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ── Paths ─────────────────────────────────────────────────────────────────────
FINE_TUNE_DIR  = REPO_ROOT / 'fine_tuning'
S4_CORPUS_PATH = FINE_TUNE_DIR / 's4_corpus.csv'
PAIRS_PATH     = FINE_TUNE_DIR / 'training_pairs.csv'
MODEL_BASE     = 'sentence-transformers/all-MiniLM-L6-v2'
MODEL_FT       = str(FINE_TUNE_DIR / 'minilm_finetuned')
RESULTS_DIR    = FINE_TUNE_DIR / 'Results'

os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(S4_CORPUS_PATH), f'MISSING: {S4_CORPUS_PATH} — run FinS4_BuildCorpus.ipynb first!'
assert os.path.exists(PAIRS_PATH),     f'MISSING: {PAIRS_PATH} — run FinS4_BuildCorpus.ipynb first!'
assert os.path.exists(MODEL_FT),       f'MISSING: {MODEL_FT} — run FinMiniLM_FineTune.ipynb first!'

print('All paths verified OK')
print(f'Base model : {MODEL_BASE}')
print(f'Fine-tuned : {MODEL_FT}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
import sys
!{sys.executable} -m pip install -q sentence-transformers faiss-cpu matplotlib scikit-learn

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import cos_sim

print('Imports OK')

## Cell 3 — Build Eval Set from True Pairs

We hold out a **stratified 20%** of pairs per category for evaluation.
The full S4 corpus is used as the retrieval pool, making this an open-domain
retrieval task (correct chunk must be retrieved from all 14k+ chunks).

In [ ]:
from sklearn.model_selection import train_test_split

corpus_df = pd.read_csv(S4_CORPUS_PATH)
pairs_df  = pd.read_csv(PAIRS_PATH)

print(f'S4 corpus chunks: {len(corpus_df):,}')
print(f'Training pairs  : {len(pairs_df):,}')

# ── Exact same split as Colab fine-tuning ─────────────────────────────────────
# train_test_split + random_state=42 + stratify=category
# → eval_df = the 20% the fine-tuned model NEVER saw during training
# → both base and fine-tuned are tested on identical questions = fair comparison
_, eval_df = train_test_split(
    pairs_df,
    test_size=0.20,
    random_state=42,
    stratify=pairs_df['category']
)
eval_df = eval_df.reset_index(drop=True)

print(f'\nEval set : {len(eval_df):,} pairs (20% held-out — never seen during fine-tuning)')
print(f'Corpus   : {len(corpus_df):,} chunks (full S4 corpus — all 41 PDFs)')
print(f'\nPer-category:')
print(eval_df['category'].value_counts().to_string())

# ── Build evaluator inputs ────────────────────────────────────────────────────
queries = {
    f'q_{i:05d}': row['question']
    for i, row in eval_df.iterrows()
}
corpus = dict(zip(corpus_df['chunk_id'].tolist(), corpus_df['text'].tolist()))
relevant_docs = {
    f'q_{i:05d}': {row['chunk_id']}
    for i, row in eval_df.iterrows()
}

print(f'\nEvaluator ready:')
print(f'  Queries       : {len(queries):,}')
print(f'  Corpus pool   : {len(corpus):,}')
print(f'  True positives: {len(relevant_docs):,}')

## Cell 4 — Evaluate with InformationRetrievalEvaluator

`InformationRetrievalEvaluator` is the standard evaluation class from
`sentence-transformers`. It encodes all queries and corpus chunks, computes
cosine similarity, ranks results, and reports NDCG@10, MRR, and Recall@k.

In [ ]:
def evaluate_model(model_path, queries, corpus, relevant_docs, name):
    """Run InformationRetrievalEvaluator and return results dict."""
    model     = SentenceTransformer(model_path)
    evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        score_functions={'cos_sim': cos_sim},
        batch_size=128,
        show_progress_bar=True,
    )
    results = evaluator(model, output_path=str(RESULTS_DIR))
    return results

print('evaluate_model() defined — ready to run Cell 5')

## Cell 5 — Base vs Fine-Tuned Comparison

Evaluate both models and print a side-by-side comparison table with
improvement deltas.

In [ ]:
print('='*60)
print('Evaluating BASE model...')
print('='*60)
results_base = evaluate_model(
    model_path=MODEL_BASE,
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name='base',
)

print('\n' + '='*60)
print('Evaluating FINE-TUNED model...')
print('='*60)
results_ft = evaluate_model(
    model_path=MODEL_FT,
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name='finetuned',
)

# ── Extract metrics by exact suffix matching ───────────────────────────────────
def get_metric(results, suffix):
    """Find metric by key ending — works regardless of name prefix."""
    for k, v in results.items():
        if k.lower().endswith(suffix.lower()):
            return round(float(v), 4)
    return None

m_base = {
    'NDCG@10'  : get_metric(results_base, 'cos_sim_ndcg@10'),
    'MRR@10'   : get_metric(results_base, 'cos_sim_mrr@10'),
    'Recall@10': get_metric(results_base, 'cos_sim_recall@10'),
}
m_ft = {
    'NDCG@10'  : get_metric(results_ft, 'cos_sim_ndcg@10'),
    'MRR@10'   : get_metric(results_ft, 'cos_sim_mrr@10'),
    'Recall@10': get_metric(results_ft, 'cos_sim_recall@10'),
}

print('\n' + '='*58)
print(f'  {"Metric":<14} {"Base MiniLM":>12}  {"Fine-Tuned":>12}  {"Delta":>8}')
print('='*58)
for metric in ['NDCG@10', 'MRR@10', 'Recall@10']:
    base_v = m_base.get(metric)
    ft_v   = m_ft.get(metric)
    if base_v is None or ft_v is None:
        print(f'  {metric:<14}  {"N/A":>12}  {"N/A":>12}')
        continue
    delta = ft_v - base_v
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '=')
    print(f'  {metric:<14} {base_v:>12.4f}  {ft_v:>12.4f}  {arrow} {abs(delta):.4f}')
print('='*58)

winner = 'Fine-Tuned MiniLM' if (m_ft.get('NDCG@10') or 0) > (m_base.get('NDCG@10') or 0) else 'Base MiniLM'
print(f'\nWinner: {winner}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metric_names = ['NDCG@10', 'MRR@10', 'Recall@10']
base_vals    = [m_base.get(m, 0) for m in metric_names]
ft_vals      = [m_ft.get(m, 0)   for m in metric_names]

x     = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars_base = ax.bar(x - width/2, base_vals, width, label='Base MiniLM',    color='#4C72B0', alpha=0.85)
bars_ft   = ax.bar(x + width/2, ft_vals,   width, label='Fine-Tuned MiniLM', color='#DD8452', alpha=0.85)

# Value labels on bars
for bar in bars_base:
    ax.annotate(f'{bar.get_height():.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=9)
for bar in bars_ft:
    ax.annotate(f'{bar.get_height():.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Base vs Fine-Tuned MiniLM — Financial Document Retrieval', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, max(max(base_vals), max(ft_vals)) * 1.2)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
chart_path = RESULTS_DIR / 'minilm_eval_comparison.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved to: {chart_path}')

# ── Save JSON results ─────────────────────────────────────────────────────────
eval_output = {
    'eval_set_size'  : len(queries),
    'corpus_size'    : len(corpus),
    'base_model'     : MODEL_BASE,
    'finetuned_model': MODEL_FT,
    'metrics': {
        'base'      : m_base,
        'finetuned' : m_ft,
        'deltas'    : {
            m: round(m_ft.get(m, 0) - m_base.get(m, 0), 4)
            for m in metric_names
        },
    },
    'winner': winner,
}

results_json_path = RESULTS_DIR / 'eval_results.json'
with open(results_json_path, 'w') as f:
    json.dump(eval_output, f, indent=2)

print(f'Results JSON saved to: {results_json_path}')
print(f'\nWinner: {winner}')